# LTLf Discrete SAC training

This notebook downloads the current modular sources, lets you configure the temporal task and training parameters, runs d3rlpy's Discrete SAC, and packages the results.

Enable **Internet** in the Kaggle notebook settings. A GPU is optional.

## 1. Install dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq mona graphviz swig
%pip install -q "d3rlpy>=2.8,<3" "gymnasium[box2d]>=1.0,<2" "ltlf2dfa>=1.0.2,<2" "graphviz>=0.20,<1" "matplotlib>=3.8,<4" "pandas>=2,<4"

import d3rlpy
import gymnasium
import torch

print(f"d3rlpy: {d3rlpy.__version__}")
print(f"Gymnasium: {gymnasium.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Create the working directory

In [ ]:
from pathlib import Path
import os
import urllib.request

WORK_DIR = Path("/kaggle/working/discrete_sac")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

SOURCE_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "matteoventali/Honour_Program/main/discrete_sac"
)

def download_source(filename):
    """Download one project source file from the main branch."""
    destination = WORK_DIR / filename
    urllib.request.urlretrieve(f"{SOURCE_BASE_URL}/{filename}", destination)
    print(f"Downloaded {filename}")

print(f"Working directory: {WORK_DIR}")

## 3. Source files

Each project source has its own cell. Re-run these cells whenever the repository sources change.

### `abstract_mdps.py`

In [ ]:
download_source("abstract_mdps.py")

### `automaton_validator.py`

In [ ]:
download_source("automaton_validator.py")

### `utils.py`

In [ ]:
download_source("utils.py")

### `trainer.py`

In [ ]:
download_source("trainer.py")

## 4. Configure `trajectory.json`

Edit the formula, abstract-grid dimensions, reward, and waypoint coordinates below.

In [ ]:
import json

trajectory = {
    "formula": "F(wp1 & X(F(g1) & G((!wp2) | F(g2))))",
    "grid_w": 12,
    "grid_h": 12,
    "gamma": 0.99,
    "goal_reward": 10000,
    "waypoints_dict": {
        "wp1": [2, 9],
        "g1": [8, 8],
        "wp2": [5, 6],
        "g2": [9, 3],
    },
}

with open("trajectory.json", "w", encoding="utf-8") as config_file:
    json.dump(trajectory, config_file, indent=4)

print(json.dumps(trajectory, indent=4))

## 5. Configure training

These values map directly to the command-line parameters exposed by `trainer.py`.

In [ ]:
STEPS = 250_000
USE_SHAPING = True
SHAPING_SCALE = 1.0

LEARNING_RATE = 3e-4
HIDDEN_SIZES = [128, 128]
INITIAL_TEMPERATURE = 0.1
BATCH_SIZE = 64
BUFFER_SIZE = 100_000
RANDOM_STEPS = 5_000
TARGET_UPDATE_INTERVAL = 8_000
UPDATE_INTERVAL = 1
UPDATES_PER_INTERVAL = 1
STEPS_PER_EPOCH = 10_000

LOG_INTERVAL = 100
PLOT_WINDOW = 50
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu:0"

print(f"Training steps: {STEPS:,}")
print(f"Reward shaping: {USE_SHAPING}")
print(f"Device: {DEVICE}")

## 6. Validate the sources

In [ ]:
import subprocess

SOURCE_FILES = [
    "abstract_mdps.py",
    "automaton_validator.py",
    "utils.py",
    "trainer.py",
]
subprocess.run(["python", "-m", "py_compile", *SOURCE_FILES], check=True)
print("All source files compiled successfully.")

## 7. Run training

In [ ]:
command = [
    "python", "trainer.py",
    "--steps", str(STEPS),
    "--config", "trajectory.json",
    "--shaping-scale", str(SHAPING_SCALE),
    "--learning-rate", str(LEARNING_RATE),
    "--hidden-sizes", *map(str, HIDDEN_SIZES),
    "--initial-temperature", str(INITIAL_TEMPERATURE),
    "--batch-size", str(BATCH_SIZE),
    "--buffer-size", str(BUFFER_SIZE),
    "--random-steps", str(RANDOM_STEPS),
    "--target-update-interval", str(TARGET_UPDATE_INTERVAL),
    "--update-interval", str(UPDATE_INTERVAL),
    "--updates-per-interval", str(UPDATES_PER_INTERVAL),
    "--steps-per-epoch", str(STEPS_PER_EPOCH),
    "--device", DEVICE,
    "--log-interval", str(LOG_INTERVAL),
    "--plot-window", str(PLOT_WINDOW),
]
command.append("--use-shaping" if USE_SHAPING else "--no-use-shaping")

environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
print("Running:", " ".join(command), flush=True)

process = subprocess.Popen(
    command,
    cwd=WORK_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for output_line in process.stdout:
    print(output_line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

## 8. Package outputs

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/discrete_sac_outputs.zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for directory_name in ("results", "img"):
        directory = WORK_DIR / directory_name
        if directory.exists():
            for path in sorted(directory.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(WORK_DIR))
    archive.write(WORK_DIR / "trajectory.json", "trajectory.json")

print(f"Archive ready: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")